# Coding Exercise: Build a Multi-Tool Agent

In this exercise, you will build an agent that uses multiple tools to answer different types of questions. The exercise has two parts.

In **Part A**, you will add web search to a LangGraph agent following guided steps similar to what you saw in Module 7. In **Part B**, you will research and implement a tool you have not been taught: a weather API using OpenWeatherMap. Part B is intentionally less guided. You will read documentation, figure out the integration, and test it yourself before adding it to your agent.

This mirrors how tool integration works in real projects, where you rarely get step-by-step instructions for every tool you need.

**Fill in all TODOs.**

---
## Setup

Run the cell below to install the required packages. You may see some warnings or dependency messages; these are safe to ignore.

In [1]:
# Install required packages (this may take a minute, and you may ignore the errors)
!pip install -qU langchain-google-genai langchain-tavily tavily-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.3 MB/s eta 0:00:00


In [2]:
# Configure your API keys
# You should already have GOOGLE_API_KEY saved in your Colab secrets.
# You will also need a TAVILY_API_KEY. If you do not have one yet:
#   1. Go to https://tavily.com and create a free account
#   2. Copy your API key
#   3. In Colab, click the key icon on the left sidebar
#   4. Add a new secret named TAVILY_API_KEY with your key as the value
# See the "Step-by-Step Web Seach Integration" page on Canvas if you are
# having trouble

import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
print("API keys configured successfully!")

API keys configured successfully!


---
## Part A: Implement Web Search (Guided)

In this part, you will create a web search tool and connect it to a LangGraph agent. This follows the same pattern you saw in the Module 7 lessons.

**Your tasks:**
1. Create the LLM and the search tool
2. Create an agent with a system prompt and the search tool
3. Test the agent with queries that require current information

### Step 1: Create the LLM and Search Tool

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_tavily import TavilySearch
from langchain.agents import create_agent

# TODO: Create the LLM with model gemini-2.0-flash
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")# YOUR CODE HERE

# TODO: Create the search tool using TavilySearch
# Use: max_results=5, search_depth="basic", include_answer=True, include_raw_content=False
search_tool = TavilySearch(
    max_results=5,
    search_depth="basic",
    include_answer=True,
    include_raw_content=False
)

### Step 2: Create the Agent

Create an agent with a system prompt that tells it to use web search for current information. Use `create_agent` with the LLM and the search tool.

In [4]:
# TODO: Write a system prompt for your search agent.
# Give it a name and a personality. Make it someone you would actually
# want to talk to, like a curious friend who loves digging into things,
# a no-nonsense news desk editor, or whatever feels right to you.
# Then instruct it to use web search for current events, recent news,
# and anything that might have changed since its training cutoff.
agent_prompt =  """You are a Tavy, a friendly and helpful research assistant.

When answering questions about current events, recent news, or anything that
might have changed after your training data, ALWAYS use the web search tool
first. Do not rely on your training data for:
- Award shows, elections, or recent events
- Current prices, statistics, or rankings
- News from the past year
- Anything the user indicates is "recent" or "latest"

Search first, then answer based on what you find."""


# TODO: Create the agent using create_agent with the LLM, the search tool, and your prompt
search_agent = create_agent(
    model=llm,
    tools=[search_tool],
    system_prompt=agent_prompt
)# YOUR CODE HERE

### Step 3: Test the Search Agent

Run the test queries below. For each one, verify two things: (1) the agent uses the search tool rather than answering from memory, and (2) the answer is accurate and current. If the agent answer is incorrect, go back and change your agent_prompt.

In [5]:
# Test query 1
test_query_1 = "What were the major news stories today?"

for chunk in search_agent.stream(
    {"messages": [{"role": "human", "content": test_query_1}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'e097a3e2-87b4-4c47-a9e9-d1550ee23431', 'name': 'tavily_search', 'args': {'time_range': 'day', 'query': 'major news stories today', 'topic': 'news'}}]

Step: tools
Content: [{'type': 'text', 'text': '{"query": "major news stories today", "follow_up_questions": null, "answer": "Major news stories today include Wendy\'s closing hundreds of underperforming US restaurants after a weak fourth quarter, a major winter storm expected to cause travel disruptions in Northern California, and an analysis showing that Americans bear nearly all the costs of tariffs. Additionally, Miami\'s last-minute 8-point run secured a narrow 77-76 victory over North Carolina State in a college basketball game.", "images": [], "results": [{"url": "https://www.newser.com/story/383728/after-limp-q4-wendys-shutters-hundreds-of-restaurants.html", "title": "You\'ll Have to Look Harder for a Frosty Now - Newser", "score": 0.5, "published_date": "Sat, 14 Feb 2026 23:10:0

In [6]:
# Test query 2
test_query_2 = "What is the current weather in New York City?"

for chunk in search_agent.stream(
    {"messages": [{"role": "human", "content": test_query_2}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'text', 'text': 'I am sorry, I cannot directly provide the current weather in New York City. However, you can easily find this information by using online weather services or apps.'}]



In [7]:
# Test query 3
test_query_3 = "Who is playing in the Super Bowl in 2026?"

for chunk in search_agent.stream(
    {"messages": [{"role": "human", "content": test_query_3}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'bc23a8a3-e8c0-46ad-9364-1bfff987b14a', 'name': 'tavily_search', 'args': {'start_date': '2024-01-01', 'query': 'Super Bowl 2026 teams'}}]

Step: tools
Content: [{'type': 'text', 'text': '{"query": "Super Bowl 2026 teams", "follow_up_questions": null, "answer": "Super Bowl 2026 featured the Seattle Seahawks against the New England Patriots. The game took place on February 12, 2026. Bad Bunny performed at the halftime show.", "images": [], "results": [{"url": "https://sports.yahoo.com/articles/super-bowl-2026-teams-date-025455875.html", "title": "When is Super Bowl 2026? Teams, date, time, TV channel, halftime ...", "content": "# When is Super Bowl 2026? The New England Patriots and Seattle Seahawks played each other in the Super Bowl in 2015. ## Who will play in Super Bowl 2026? The New England Patriots will play the Seattle Seahawks in Super Bowl 60. ## Have the Patriots won a Super Bowl? The Patriots have won six Super Bowls, tied for 

---
## Part B: Research and Implement a Weather Tool (Discovery)

Now you will implement a tool you have not been taught: a weather API tool using OpenWeatherMap. If this feels slower or messier than Part A, that is expected. This is what real-world tool integration looks like.

### What you are given

- A link to the OpenWeatherMap API documentation
- A link to the LangChain integrations page (to check if a built-in integration exists)
- Minimal starter code

### What you must do

1. **Get an API key**: Go to [https://openweathermap.org/api](https://openweathermap.org/api), create a free account, and get your API key. Save it in Colab secrets as `OPENWEATHERMAP_API_KEY`.
2. **Read the documentation**: Understand how the API works, what inputs it needs, and what it returns.
3. **Implement the weather tool**: Find the LangChain documentation for OpenWeatherMap integration and write code to implement the tool.
5. **Add it to your agent**: Create a multi-tool agent with both web search and the weather tool.

### Tips

- The free tier of OpenWeatherMap is sufficient for this exercise.
- New API keys can take a few minutes to activate. If you get an authentication error right after creating your key, wait a couple of minutes and try again.
- If you cannot find a LangChain integration, you can wrap the API yourself using the `@tool` decorator and the `requests` library. You saw this pattern in Module 3.
- Test the tool in isolation before integrating it. This makes debugging much easier.

### Step 1: Set Up Your OpenWeatherMap API Key

In [8]:
# TODO: Add your OpenWeatherMap API key to Colab secrets
# Then configure it here:

os.environ["OPENWEATHERMAP_API_KEY"] = userdata.get("OPENWEATHERMAP_API_KEY")
print("OpenWeatherMap API key configured!")

OpenWeatherMap API key configured!


### Step 2: Research and Implement the Weather Tool

You are intentionally not given step-by-step code here. Use the documentation links above and the tool-building patterns you learned in Module 7 to implement this.

In [9]:
!pip install -qU pyowm langchain_community

from langchain_community.utilities.openweathermap import OpenWeatherMapAPIWrapper

# Install any additional packages you need for the weather tool
# YOUR CODE HERE

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 91.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [17]:
# TODO: Implement the weather tool
from langchain.tools import tool

weather_wrapper = OpenWeatherMapAPIWrapper()

@tool
def get_weather(location: str) -> str:
    """Get current weather information for a location.

    Args:
        location: Location in format 'City,CountryCode' (e.g., 'London,GB', 'New York,US')

    Returns:
        Current weather information for the specified location
    """
    return weather_wrapper.run(location)


### Step 3: Test the Weather Tool in Isolation

Before connecting the weather tool to your agent, test it directly. Make sure it returns accurate weather data.

In [19]:
# TODO: Test your weather tool with at least two different cities
# For example: New York, Los Angeles, London
# Verify the output looks correct
weather_data = get_weather.invoke("New York,US")
print(weather_data)

weather_data = get_weather.invoke("Los Angeles,US")
print(weather_data)
# YOUR CODE HERE

In New York,US, the current weather is as follows:
Detailed status: overcast clouds
Wind speed: 4.63 m/s, direction: 50°
Humidity: 63%
Temperature: 
  - Current: 4.14°C
  - High: 5.45°C
  - Low: 2.76°C
  - Feels like: 0.45°C
Rain: {}
Heat index: None
Cloud cover: 100%
In Los Angeles,US, the current weather is as follows:
Detailed status: scattered clouds
Wind speed: 2.57 m/s, direction: 250°
Humidity: 65%
Temperature: 
  - Current: 16.45°C
  - High: 17.71°C
  - Low: 14.84°C
  - Feels like: 15.85°C
Rain: {}
Heat index: None
Cloud cover: 40%


---
## Part C: Build and Test the Multi-Tool Agent

Now combine both tools into a single agent. Write a system prompt that helps the agent decide which tool to use for each query.

In [21]:
# TODO: Write a system prompt for your multi-tool agent.
# The prompt should explain when to use web search and when to use the weather tool.
# Think about: what types of questions should go to each tool?
multi_tool_prompt = """You are a helpful AI assistant with access to current information.

You have two tools:
1. **Web search (Tavily)**: Use this for current information, news, facts, research, recommendations, or any topic that requires up-to-date knowledge or multiple sources.

2. **Weather tool (OpenWeather API)**: Use this specifically for weather information. Requires location in format: "City,CountryCode" (e.g., "London,GB", "New York,US", "Tokyo,JP").

Examples:
* "What's the weather in Paris?" → Use weather tool with "Paris,FR"
* "Should I bring an umbrella to Seattle tomorrow?" → Use weather tool with "Seattle,US"
* "What are the top tourist attractions in Tokyo?" → Use web search only
* "What's the weather like in London and what should I do there?" → Use weather tool for "London,GB", then web search for activities
* "Tell me about climate change" → Use web search only
* "Is it raining in Berlin right now?" → Use weather tool with "Berlin,DE"

Important guidelines:
- For weather queries, always use the weather tool with proper location format
- For everything else (news, facts, recommendations, research), use web search
- If you're unsure about the country code, use web search first to clarify the location
- Always use tools rather than relying on potentially outdated knowledge
- If a question requires both tools, use them in sequence

Keep your responses helpful, accurate, and concise."""

# TODO: Create the multi-tool agent with both tools
multi_tool_agent = create_agent(
    model = llm,
    tools = [search_tool, get_weather],
    system_prompt = multi_tool_prompt) # YOUR CODE HERE

### Test the Multi-Tool Agent

Run the queries below. For each one, check which tool the agent selects. The expected tool is noted in the comment.

If the agent picks the wrong tool, revisit your system prompt and your tool descriptions.

In [22]:
# Test query 1 (expected: weather tool)
query_1 = "What's the weather in Chicago?"

for chunk in multi_tool_agent.stream(
    {"messages": [{"role": "human", "content": query_1}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': '939a8782-adc9-4823-9bed-917a460be5e2', 'name': 'get_weather', 'args': {'location': 'Chicago,US'}}]

Step: tools
Content: [{'type': 'text', 'text': 'In Chicago,US, the current weather is as follows:\nDetailed status: clear sky\nWind speed: 4.12 m/s, direction: 290°\nHumidity: 52%\nTemperature: \n  - Current: 13.99°C\n  - High: 15.12°C\n  - Low: 13.04°C\n  - Feels like: 12.8°C\nRain: {}\nHeat index: None\nCloud cover: 0%'}]

Step: model
Content: [{'type': 'text', 'text': 'The current weather in Chicago, US is clear with a temperature of 13.99°C, but it feels like 12.8°C. The wind is blowing at 4.12 m/s from 290 degrees. The humidity is 52% and there is no rain. The high for today is 15.12°C and the low is 13.04°C. There is 0% cloud cover.'}]



In [23]:
# Test query 2 (expected: web search)
query_2 = "What are today's top news stories?"

for chunk in multi_tool_agent.stream(
    {"messages": [{"role": "human", "content": query_2}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': '4c63b3ac-7d84-4f93-8dc9-7ac43de2a533', 'name': 'tavily_search', 'args': {'topic': 'news', 'query': "today's top news stories"}}]

Step: tools
Content: [{'type': 'text', 'text': '{"query": "today\'s top news stories", "follow_up_questions": null, "answer": "Today\'s top news stories include criticism of Andhra Pradesh\'s agriculture budget by the YSRCP, which alleges it failed to address farmers\' distress. The Communist Party of India (CPI) also criticized the budget as a self-congratulatory exercise lacking balance between welfare and development. Additionally, the Enforcement Directorate (ED) has summoned former TDB secretary S. Jayasree and intermediary Kalpesh in connection with the Sabarimala gold theft case. Sports betting highlights include the best bets and odds for college basketball games and NBA All-Star Weekend events.", "images": [], "results": [{"url": "https://www.thehindu.com/news/national/andhra-pradesh/agriculture-bud

In [24]:
# Test query 3 (expected: weather tool)
query_3 = "What's the temperature in Tokyo right now?"

for chunk in multi_tool_agent.stream(
    {"messages": [{"role": "human", "content": query_3}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'd428841f-aa38-4867-9102-7d9586e3fa53', 'name': 'get_weather', 'args': {'location': 'Tokyo,JP'}}]

Step: tools
Content: [{'type': 'text', 'text': 'In Tokyo,JP, the current weather is as follows:\nDetailed status: clear sky\nWind speed: 3.09 m/s, direction: 340°\nHumidity: 67%\nTemperature: \n  - Current: 9.62°C\n  - High: 10.91°C\n  - Low: 7.53°C\n  - Feels like: 8.0°C\nRain: {}\nHeat index: None\nCloud cover: 0%'}]

Step: model
Content: [{'type': 'text', 'text': 'The temperature in Tokyo is currently 9.62°C, but it feels like 8.0°C. The high for today is 10.91°C and the low is 7.53°C. The sky is clear.'}]



In [25]:
# Test query 4 (expected: web search)
query_4 = "Who won the most Grammys in 2026?"

for chunk in multi_tool_agent.stream(
    {"messages": [{"role": "human", "content": query_4}]},
    stream_mode="updates"
):
    for step, data in chunk.items():
        print(f"Step: {step}")
        print(f"Content: {data['messages'][-1].content_blocks}")
        print()

Step: model
Content: [{'type': 'tool_call', 'id': 'b445449f-ce17-49b6-973d-ebb356cece38', 'name': 'tavily_search', 'args': {'query': 'who won the most grammys in 2026'}}]

Step: tools
Content: [{'type': 'text', 'text': '{"query": "who won the most grammys in 2026", "follow_up_questions": null, "answer": "Kendrick Lamar won the most awards at the 2026 Grammys with 26 wins. He set a record as the most awarded rapper in the event\'s history. Other notable winners included Jelly Roll and Finneas O\'Connell.", "images": [], "results": [{"url": "https://www.instagram.com/p/DUPka9QjZEv/", "title": "The 2026 Grammys was one for the record books. Kendrick Lamar ...", "content": "Kendrick Lamar, who led the 2026 nominations with nine nods, became the most awarded rapper in the event\'s history with 26 total wins—beating", "score": 0.8565368, "raw_content": null}, {"url": "https://www.billboard.com/lists/2026-grammys-winners-list/", "title": "Here Are the 2026 Grammys Winners: Full List - Billboa

---
## Reflection

Answer the following question in 2 to 3 sentences:

**What was your process for figuring out the weather tool? Describe the steps you took from reading documentation to working implementation.**

My implementation followed a systematic approach: I first reviewed the OpenWeatherMap website to verify the free tier availability (1,000 calls/day) and studied their API documentation to understand the input format (City,CountryCode) and output structure. I then researched LangChain's existing integrations and found their pre-built OpenWeatherMapAPIWrapper utility with comprehensive documentation. Following the patterns established in Module 2, I implemented the weather tool using the @tool decorator to ensure compatibility with the LangChain framework. Finally, I integrated both the search tool (Tavily) and weather tool into a unified multi-tool agent, crafting a clear system prompt to define when each tool should be invoked based on the user's query type.

---
**To submit:** Download this notebook as a .ipynb file (File > Download > Download .ipynb) and upload it to Canvas.